In [2]:
# interactive trajectory intercept minimizer
import importlib

import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, FloatSlider, Layout
import numpy as np
from orbitengine.body import Body
import orbitengine.engine as oe
import astropy.units as u
from scipy.spatial.transform import Rotation as R
from scipy.optimize import minimize
import time

importlib.reload(oe)
#importlib.reload(Body)

# Define a function that creates a plot
m0 = 5000.0*u.kg
state_init = Body.State(
    np.array([0,oe.EARTH_RADIUS_KM, 0])*u.km, 
    oe.V_ZERO, 
    m0, 
    parent_axis_angle=oe.EARTH_AXIS_ANGLE)

def target_dist(x, thrust_angle, state_init, state_target):
    velocity, t_flight = x
    rot = R.from_euler('z', thrust_angle)
    v = np.array([velocity, 0, 0])*u.km/u.s

    state_maneuver = Body.State(state_init.position, 
                     rot.apply(v)*u.km/u.s, 
                     state_init.mass)

    sm = state_maneuver.propagate(oe.EARTH_K, t_flight*u.s)
    st = state_target.propagate(oe.EARTH_K, t_flight*u.s)

    dp = np.linalg.norm(st.position - sm.position).value

    return dp


def f(target_angle, thrust_angle, t_flight, velocity):
    rot_target = R.from_euler('z', target_angle)
    state_target = Body.State(rot_target.apply(state_init.position)*u.km, 
                              oe.V_ZERO, 
                              m0,
                              parent_axis_angle=oe.EARTH_AXIS_ANGLE)


    x0 = [velocity, t_flight]
    ts_start = time.time()

    res = minimize(target_dist, 
                   x0, 
                   args=(thrust_angle, state_init, state_target))
    ts_stop = time.time()
    print(f"Optimization time: {ts_stop - ts_start:.3f} s")

    # print(f"x0: {x0}")
    # print(f"res.x: {res.x}")
    # print(f"res.fun: {res.fun}")

    # original maneuver
    rot = R.from_euler('z', thrust_angle)
    v = np.array([velocity, 0, 0])*u.km/u.s
    state_m1 = Body.State(state_init.position, rot.apply(v)*u.km/u.s, state_init.mass)

    ts1 = np.linspace(0, t_flight*u.s, 100)
    sm1 = state_m1.propagate(oe.EARTH_K, ts1)

    fig, axs = plt.subplots(1, 1, figsize=(5, 5))

    axs.plot([s.position[0].value for s in sm1], [s.position[1].value for s in sm1], color='b')
    axs.plot(sm1[-1].position[0].value, sm1[-1].position[1].value, 'bo')

    # optimized maneuver
    velocity, t_flight = res.x
    print(f"Optimized t_flight: {t_flight*u.s:.2f}")
    print(f"Optimized velocity: {velocity*u.km/u.s:.2f}")
    rot = R.from_euler('z', thrust_angle)
    v = np.array([velocity, 0, 0])*u.km/u.s
    state_m2 = Body.State(state_init.position, rot.apply(v)*u.km/u.s, state_init.mass, T=oe.TEMP_EARTH)

    #plot trajectory
    ts2 = np.linspace(0, t_flight*u.s, 100)
    sm2 = state_m2.propagate(oe.EARTH_K, ts2)
    axs.plot([s.position[0].value for s in sm2], [s.position[1].value for s in sm2],color='g')
#    axs.plot(sm2[-1].position[0].value, sm2[-1].position[1].value, 'go') # should overlap with target

    stp = state_target.propagate(oe.EARTH_K, t_flight*u.s).position.value
    axs.plot(stp[0], stp[1], 'ro')
    axs.add_artist(plt.Line2D((0, stp[0]), (0, stp[1]), color='r', linestyle='dotted'))

    axs.plot(state_init.position[0].value, state_init.position[1].value, 'go')

    dr = np.linalg.norm(stp - sm2[-1].position.value)*u.km
    print(f"Distance to target: {dr:.3f}")
 
    circle = plt.Circle((0, 0), oe.EARTH_RADIUS_KM, color='b', fill=False, linestyle='dotted')
    axs.add_artist(circle)
    axs.add_artist(plt.Line2D((0, state_init.position[0].value), (0, state_init.position[1].value), color='g', linestyle='dotted'))
    axs.set_aspect('equal', adjustable='box')
    plt.show()

    plt.plot(ts2, [s.temperature.value for s in sm2])
    plt.show()

target_angle_slider = FloatSlider(min=-np.pi, max=np.pi, value=-0.002,step=0.01, layout=Layout(width='800px'))
thrust_angle_slider = FloatSlider(min=0, max=np.pi, value=0.5,step=0.01, layout=Layout(width='800px'))
t_flight_slider = IntSlider(min=0, max=5000,value=15, layout=Layout(width='800px'))
velocity_slider = FloatSlider(min=0, max=10, step=0.01, value=0.17, layout=Layout(width='800px'))

# Create a slider that controls the frequency of the sine wave
interact(f, 
         target_angle=target_angle_slider, 
         thrust_angle=thrust_angle_slider,
         t_flight=t_flight_slider, 
         velocity=velocity_slider)



ImportError: module Body not in sys.modules